In [ ]:
import transformers
from transformers import PreTrainedModel, PreTrainedTokenizer
from tqdm.auto import tqdm
import abc

from src import utils

In [ ]:
from huggingface_hub import HfFolder

hf_token = utils.api_key_from_file("HF_KEY.txt")

HfFolder.save_token(hf_token)

In [ ]:
from src.attack import Attack
from src.embed_injector import EmbedInjector

import torch
from tqdm.auto import tqdm
from abc import ABC, abstractmethod
import pathlib
import time
from typing import Iterable, Callable


class StopCriteria:
    def __init__(
        self,
        max_epochs: int = 100,
        max_evals: int | None = None,
        max_time: float | None = None,
        target_value: float | None = None,
        patience: int | None = None,
        patience_delta: float = 1e-4,
    ):
        """
        Container for various stopping criteria.

        Args:
            max_epochs: Maximum number of epochs.
            max_evals: Maximum number of evaluation steps.
            max_time: Maximum training time in seconds.
            target_value: Target value to stop training when reached. This value should be maximized.
            patience: Number of evaluation steps without sufficient improvement.
            patience_delta: Minimum improvement delta to reset patience.

        Raises:
            ValueError: If any of the arguments are invalid.
        """
        if max_epochs <= 0:
            raise ValueError("Max epochs must be greater than 0.")
        if max_evals is not None and max_evals <= 0:
            raise ValueError("Max evals must be greater than 0.")
        if max_time is not None and max_time <= 0:
            raise ValueError("Max time must be greater than 0.")
        if patience is not None and patience <= 0:
            raise ValueError("Patience must be greater than 0.")
        if patience_delta < 0:
            raise ValueError("Patience delta must be greater than or equal to 0.")

        self.max_epochs = max_epochs
        self.max_evals = max_evals
        self.max_time = max_time
        self.target_value = target_value
        self.patience = patience
        self.patience_delta = patience_delta

        # Internal state
        self._epoch = 0
        self._total_evals = 0
        self._start_time = time.time()
        self._best_value = -float("inf")
        self._patience_counter = 0
        self.reset()

    def get_hparams(self) -> dict:
        return {
            "stop/max_epochs": self.max_epochs,
            "stop/max_evals": self.max_evals,
            "stop/max_time": self.max_time,
            "stop/target_value": self.target_value,
            "stop/patience": self.patience,
            "stop/patience_delta": self.patience_delta,
        }

    def reset(self) -> None:
        """Reset internal state."""
        self._epoch = 0
        self._total_evals = 0
        self._start_time = time.time()
        self._best_value = -float("inf")
        self._patience_counter = 0

    def update(self, epoch: int, value: float) -> None:
        """Update internal state with new metrics."""
        self._epoch = epoch
        self._total_evals += 1

        if (value - self._best_value) >= self.patience_delta:
            self._best_value = value
            self._patience_counter = 0
        else:
            self._patience_counter += 1

    def should_stop(self) -> bool:
        """Check if any stopping condition is met."""
        if self.target_value is not None and self._best_value >= self.target_value:
            print(f"Stopping: Target value reached :: ({self.target_value})")
            return True

        if self._epoch >= self.max_epochs:
            print(f"Stopping: Max epochs reached :: ({self.max_epochs})")
            return True

        if self.max_evals is not None and self._total_evals >= self.max_evals:
            print(f"Stopping: Max evals reached :: ({self.max_evals})")
            return True

        if self.patience is not None and self._patience_counter >= self.patience:
            print(f"Stopping: Patience exceeded :: ({self.patience})")
            return True

        if self.max_time is not None and (time.time() - self._start_time) > self.max_time:
            print(f"Stopping: Max time reached :: ({self.max_time} sec)")
            return True

        return False


class IMLAttack:
    def __init__(
        self,
        embed_injector: EmbedInjector,
        internal_attack: Attack,
        mixed_precision: bool = True,
    ):
        self.embed_injector = embed_injector
        self.device = embed_injector.device
        self.internal_attack = internal_attack
        self.mixed_precision = mixed_precision

    def compute_logits(self, embed_dict: dict, adv_embed: torch.Tensor) -> torch.Tensor:

        inj_embed = self.embed_injector.inject_embedding(
            inp_embeds=embed_dict["inputs_embeds"],
            adv_embeds=adv_embed,
            adv_mask=embed_dict["adv_mask"],
        )

        sample_result = self.embed_injector.forward(
            inj_embed,
            attention_mask=embed_dict["attention_mask"],
        )

        logits, _ = self.internal_attack.align_preds(
            sample_result.logits,
            input_ids=embed_dict["input_ids"],
            target_mask=embed_dict["target_mask"],
        )

        return logits

    def fit(
        self,
        dl_train: Iterable,
        dl_eval: Iterable | None = None,
        stop: StopCriteria | None = None,
    ) -> torch.Tensor:

        if dl_eval is None:
            dl_eval = dl_train

        if stop is None:
            stop = StopCriteria()

        # initialize adv embedding
        univ_embed: torch.Tensor = torch.randn(
            size=(1, self.internal_attack.num_tokens, self.embed_injector.embed_dim),
            device=self.device,
            requires_grad=True,
        )

        scaler = torch.GradScaler(enabled=self.mixed_precision)
        optim = torch.optim.AdamW([univ_embed], lr=1e-3, maximize=True)  # very important to maximize

        with tqdm(range(stop.max_epochs), desc="Epochs") as epoch_pbar:
            for epoch in epoch_pbar:
                for batch in tqdm(dl_train, desc="Batch", leave=False):
                    input_text, target_text = batch

                    optim.zero_grad()
                    embed_dict = self.embed_injector.embed_text(input_text, target_text)
                    univ_embed_broad = torch.broadcast_to(univ_embed, (len(input_text), *univ_embed.shape[1:]))

                    sample_embed = self.internal_attack.fit(input_text, target_text, init_embedding=univ_embed_broad)

                    with torch.autocast(device_type=self.device.type, enabled=self.mixed_precision):

                        # compute logits for per-sample and univ embedding
                        univ_pred_logits = self.compute_logits(embed_dict, univ_embed_broad)
                        with torch.inference_mode():
                            sample_pred_logits = self.compute_logits(embed_dict, sample_embed)

                        # compute loss
                        univ_pred_logits = univ_pred_logits.view(-1, univ_pred_logits.size(-1))
                        sample_pred_logits = sample_pred_logits.view(-1, sample_pred_logits.size(-1))
                        loss = torch.cosine_similarity(univ_pred_logits, sample_pred_logits, dim=-1).mean()

                    scaler.scale(loss).backward()
                    scaler.step(optim)
                    scaler.update()

                    epoch_pbar.set_postfix({"loss": loss.item()})

        return univ_embed.detach()

In [ ]:
import pandas as pd


def build_dataframe(data_path: str) -> pd.DataFrame:
    data = pd.read_json(data_path)
    data = pd.DataFrame.from_records(data["data"])
    data = data[["behavior", "default_target"]]
    data = data.rename(columns={"behavior": "prompt", "default_target": "output"})
    return data


data_train = build_dataframe("circuit-breakers-eval/data/harmbench_test_std.json")

ds_train = data_train[["prompt", "output"]]

# turn ds_train and ds_eval into dl_train and dl_eval
# by crating batches of size 100, each batch is a tuple of (input_text, target_text)

dl_train = []

batch_size = 10

for i in range(0, len(ds_train), batch_size):
    input_text = ds_train.iloc[i : i + batch_size]["prompt"].tolist()
    target_text = ds_train.iloc[i : i + batch_size]["output"].tolist()
    dl_train.append((input_text, target_text))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch import optim
from src.embed_injector import EmbedInjector
from src.attacks.optim_attack import OptimAttack

# model_name = "Qwen/Qwen3-0.6B"
model_name = "meta-llama/Llama-3.2-1B-Instruct"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

utils.set_seed(42)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    attn_implementation="sdpa",
)

torch.set_float32_matmul_precision("high")  # negligable effect

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

inj_model = EmbedInjector(
    model=model,
    tokenizer=tokenizer,
    num_tokens=5,
)

internal_attack = OptimAttack(
    inj_model,
    optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
    steps=30,
    silent=False,
    mixed_precision=False,
)

iml_attack = IMLAttack(
    embed_injector=inj_model,
    internal_attack=internal_attack,
    mixed_precision=False,
)

stop = StopCriteria(
    max_epochs=5,
)

univ_pert = iml_attack.fit(dl_train, stop=stop)

In [ ]:
test_prompts = [prompt for prompt, _ in ds_train[:50].itertuples(index=False)]
test_labels = [label for _, label in ds_train[:50].itertuples(index=False)]

univ_pert_broad = univ_pert.broadcast_to((len(test_prompts), *univ_pert.shape[1:]))

preds = inj_model.generate(test_prompts, univ_pert_broad, max_length=500)

for inp, lbl, pred in zip(test_prompts, test_labels, preds):
    print("Input:", inp)
    print("Label:", lbl)
    print("Pred :", pred)
    print()